# Lowercase Letters CNN (a-z) — real EMNIST

MNIST 노트북과 같은 CNN 구조로, 이번엔 진짜 손글씨 데이터셋인 EMNIST에서 소문자만 뽑아 학습한다. (이전 버전은 폰트로 찍어낸 합성 데이터였는데, 진짜 손글씨로 다시 했다.)

In [ ]:
import keras
from keras.models import Sequential, load_model
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D, Input
import numpy as np
import string

LABELS = list(string.ascii_lowercase)  # a-z, index 0-25

**데이터 준비**

EMNIST는 tensorflow-datasets로 자동 다운로드하려 했지만, 원본 서버(biometrics.nist.gov)가 이 환경에서 직접 접근이 안 돼서 브라우저로 따로 받은 `gzip.zip`을 풀어서 썼다. `byclass` 스플릿을 쓰는 이유는, `letters` 스플릿은 대문자/소문자를 같은 클래스로 합쳐버리기 때문이다. byclass는 62개 클래스(숫자 10 + 대문자 26 + 소문자 26)로 나뉘어 있어서 소문자만 따로 골라낼 수 있다 (라벨 36~61이 a~z에 대응).

In [ ]:
# byclass has 62 classes: 0-9 digits, then A-Z, then a-z.
# Label 36 = 'a' (ASCII 97), label 61 = 'z' (ASCII 122) -- see emnist-byclass-mapping.txt.
import gzip
import struct

DATA_DIR = "gzip"  # folder from the official EMNIST zip, files still .gz -- we read them straight, no need to unzip by hand
LOWER_START = 36


def read_idx_images(path_gz):
    with gzip.open(path_gz, "rb") as f:
        magic, n, rows, cols = struct.unpack(">IIII", f.read(16))
        return np.frombuffer(f.read(n * rows * cols), dtype=np.uint8).reshape(n, rows, cols)


def read_idx_labels(path_gz):
    with gzip.open(path_gz, "rb") as f:
        magic, n = struct.unpack(">II", f.read(8))
        return np.frombuffer(f.read(n), dtype=np.uint8)


def filter_lowercase(images, labels, per_class_cap, seed):
    # take an even number of samples per letter so the classes stay balanced
    rng = np.random.RandomState(seed)
    keep_x, keep_y = [], []
    for letter_idx in range(26):
        idxs = np.where(labels == LOWER_START + letter_idx)[0]
        rng.shuffle(idxs)
        idxs = idxs[:per_class_cap]
        keep_x.append(images[idxs])
        keep_y.append(np.full(len(idxs), letter_idx, dtype=np.uint8))
    return np.concatenate(keep_x), np.concatenate(keep_y)

In [ ]:
train_labels = read_idx_labels(f"{DATA_DIR}/emnist-byclass-train-labels-idx1-ubyte.gz")
train_images = read_idx_images(f"{DATA_DIR}/emnist-byclass-train-images-idx3-ubyte.gz")
test_labels = read_idx_labels(f"{DATA_DIR}/emnist-byclass-test-labels-idx1-ubyte.gz")
test_images = read_idx_images(f"{DATA_DIR}/emnist-byclass-test-images-idx3-ubyte.gz")

# 450/class keeps training fast -- byclass actually has way more than this per letter
x_train_raw, y_train_raw = filter_lowercase(train_images, train_labels, per_class_cap=450, seed=1)
x_test_raw, y_test_raw = filter_lowercase(test_images, test_labels, per_class_cap=100, seed=2)

# EMNIST images are stored rotated/transposed vs. how a person would draw them, fix that
x_train_raw = np.transpose(x_train_raw, (0, 2, 1))
x_test_raw = np.transpose(x_test_raw, (0, 2, 1))

# filter_lowercase() builds the array one letter at a time, so it's sorted by class right now.
# shuffle before training, otherwise validation_split just grabs the last letter or two.
rng = np.random.RandomState(3)
idx = rng.permutation(len(x_train_raw))
x_train_raw, y_train_raw = x_train_raw[idx], y_train_raw[idx]

print(f"train: {x_train_raw.shape}, test: {x_test_raw.shape}")

train: (11700, 28, 28), test: (2600, 28, 28)


**데이터 전처리** (MNIST.ipynb과 동일: reshape → float32/255 → one-hot)

In [ ]:
BATCH_SIZE = 128
NUM_CLASSES = 26      # a-z instead of MNIST's 0-9
NUM_EPOCHS = 8

x_train = x_train_raw.reshape(-1, 28, 28, 1).astype('float32') / 255
x_test = x_test_raw.reshape(-1, 28, 28, 1).astype('float32') / 255

y_train = keras.utils.to_categorical(y_train_raw, NUM_CLASSES)
y_test = keras.utils.to_categorical(y_test_raw, NUM_CLASSES)

**CNN 모델 구조** — MNIST.ipynb과 층 구성이 동일하다. 마지막 Dense 출력만 10 → 26.

In [ ]:
model = Sequential()
model.add(Input(shape=(28, 28, 1)))
model.add(Conv2D(32, kernel_size=(3, 3), activation='relu'))
model.add(Conv2D(64, kernel_size=(3, 3), activation='relu'))

model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))
model.add(Flatten())

model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(NUM_CLASSES, activation='softmax'))
model.summary()

Model: "sequential"
+---------------------------------+------------------------+---------------+
| Layer (type)                    | Output Shape           |       Param # |
+---------------------------------+------------------------+---------------+
| conv2d (Conv2D)                 | (None, 26, 26, 32)     |           320 |
| conv2d_1 (Conv2D)                | (None, 24, 24, 64)     |        18,496 |
| max_pooling2d (MaxPooling2D)    | (None, 12, 12, 64)     |             0 |
| dropout (Dropout)               | (None, 12, 12, 64)     |             0 |
| flatten (Flatten)               | (None, 9216)           |             0 |
| dense (Dense)                   | (None, 128)            |     1,179,776 |
| dropout_1 (Dropout)             | (None, 128)            |             0 |
| dense_1 (Dense)                 | (None, 26)             |         3,354 |
+---------------------------------+------------------------+---------------+
Total params: 1,201,946 (4.59 MB)


**컴파일 / 학습**

In [ ]:
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

history = model.fit(x_train, y_train, batch_size=BATCH_SIZE, epochs=NUM_EPOCHS,
                     verbose=2, validation_split=0.1)

Epoch 1/8
83/83 - 18s - 210ms/step - accuracy: 0.4802 - loss: 1.7904 - val_accuracy: 0.7632 - val_loss: 0.8407
Epoch 2/8
83/83 - 18s - 210ms/step - accuracy: 0.7169 - loss: 0.9534 - val_accuracy: 0.8308 - val_loss: 0.5813
Epoch 3/8
83/83 - 18s - 210ms/step - accuracy: 0.7900 - loss: 0.7059 - val_accuracy: 0.8538 - val_loss: 0.4797
Epoch 4/8
83/83 - 18s - 210ms/step - accuracy: 0.8164 - loss: 0.5972 - val_accuracy: 0.8684 - val_loss: 0.4481
Epoch 5/8
83/83 - 18s - 210ms/step - accuracy: 0.8410 - loss: 0.5073 - val_accuracy: 0.8795 - val_loss: 0.4258
Epoch 6/8
83/83 - 18s - 210ms/step - accuracy: 0.8536 - loss: 0.4625 - val_accuracy: 0.8795 - val_loss: 0.4272
Epoch 7/8
83/83 - 18s - 210ms/step - accuracy: 0.8592 - loss: 0.4317 - val_accuracy: 0.8838 - val_loss: 0.3845
Epoch 8/8
83/83 - 18s - 210ms/step - accuracy: 0.8768 - loss: 0.3870 - val_accuracy: 0.8957 - val_loss: 0.3794


**성능 평가**

In [ ]:
score = model.evaluate(x_test, y_test, verbose=0)
print('Final test accuracy: %.2f%%' % (score[1] * 100))
print('Final test loss: %.4f' % score[0])

Final test accuracy: 88.31%
Final test loss: 0.3831


MNIST(98.8%)보다 정확도가 낮게 나온다(약 88%). 숫자 10개보다 글자 26개가 헷갈릴 조합이 많아서 그렇다 (예: 소문자 i와 l, u와 v처럼 사람이 봐도 헷갈리는 글자쌍이 실제로 존재한다). 데이터도 클래스당 450장으로 MNIST보다 훨씬 적게 썼으니, 데이터를 더 많이 쓰면 정확도는 더 올라갈 것이다.

**그림판으로 직접 테스트해보기 (로컬용, 이 셀만 실행해도 됨)**

원래 MNIST.ipynb 버전은 Colab의 `google.colab.output.eval_js`로 브라우저 캔버스를 띄우는 방식이라 로컬 Jupyter에서는 돌아가지 않았다. 여기서는 대신 Tkinter로 별도 창을 띄운다 -- 파이썬 표준 라이브러리라 따로 설치할 것도 없다. 노트북을 처음부터 다 실행할 필요 없이, 저장된 모델(`lowercase_cnn_emnist.keras`, 같은 폴더에 같이 있음)을 바로 불러오기 때문에 이 셀 하나만 실행해도 그림 테스트를 할 수 있다. (Colab은 로컬 디스플레이가 없어서 Tkinter 창 자체가 안 뜨니, Colab에서 쓰려면 이전 버전의 브라우저 캔버스 방식이 필요하다.)

In [2]:
# self-contained: don't rely on the `model` variable from earlier cells,
# just load the trained model straight from disk
# %pip install opencv-python
# %pip install keras
import tkinter as tk
import string
import numpy as np
from PIL import Image, ImageDraw
import cv2
from keras.models import load_model

LABELS = list(string.ascii_lowercase)
MODEL_PATH = "lowercase_cnn_emnist.keras"  # sits next to this notebook
model = load_model(MODEL_PATH)


def crop_and_pad_image(img):
    # crop to the drawn strokes, then center on a 28x28 canvas -- same trick as the MNIST notebook
    coords = cv2.findNonZero(img)
    if coords is None:
        return None
    x, y, w, h = cv2.boundingRect(coords)
    cropped = img[y:y+h, x:x+w]

    side = max(w, h)
    square = np.zeros((side, side), dtype=np.uint8)
    x_off, y_off = (side - w) // 2, (side - h) // 2
    square[y_off:y_off+h, x_off:x_off+w] = cropped

    resized = cv2.resize(square, (20, 20), interpolation=cv2.INTER_AREA)
    final = np.zeros((28, 28), dtype=np.uint8)
    final[4:24, 4:24] = resized
    return final


class LetterDrawer:
    """Small Tkinter window: draw a letter with the mouse, hit Predict, see what the model thinks."""

    CANVAS_SIZE = 280
    BRUSH_RADIUS = 8

    def __init__(self, model):
        self.model = model
        self.root = tk.Tk()
        self.root.title("소문자 알파벳 그려보기 (a-z)")

        self.canvas = tk.Canvas(self.root, width=self.CANVAS_SIZE, height=self.CANVAS_SIZE, bg="black", cursor="crosshair")
        self.canvas.pack()
        self.canvas.bind("<B1-Motion>", self.paint)
        self.canvas.bind("<ButtonRelease-1>", self.stop_stroke)
        self.last_x, self.last_y = None, None

        # draw on-screen (Tkinter canvas) and off-screen (PIL image) at the same time,
        # the PIL copy is what we actually feed into the model
        self.image = Image.new("L", (self.CANVAS_SIZE, self.CANVAS_SIZE), color=0)
        self.draw = ImageDraw.Draw(self.image)

        btn_row = tk.Frame(self.root)
        btn_row.pack(pady=8)
        tk.Button(btn_row, text="지우기", command=self.clear).pack(side=tk.LEFT, padx=5)
        tk.Button(btn_row, text="예측하기", command=self.predict, bg="#4285F4", fg="white").pack(side=tk.LEFT, padx=5)
        tk.Button(btn_row, text="종료", command=self.root.destroy, bg="#EA4335", fg="white").pack(side=tk.LEFT, padx=5)

        self.result_label = tk.Label(self.root, text="여기에 그리고 '예측하기'를 눌러보세요", font=("Arial", 13))
        self.result_label.pack(pady=(0, 8))

    def paint(self, event):
        x, y = event.x, event.y
        if self.last_x is not None:
            self.canvas.create_line(self.last_x, self.last_y, x, y, fill="white",
                                     width=self.BRUSH_RADIUS * 2, capstyle=tk.ROUND, smooth=True)
            self.draw.line([self.last_x, self.last_y, x, y], fill=255, width=self.BRUSH_RADIUS * 2)
        self.last_x, self.last_y = x, y

    def stop_stroke(self, event):
        self.last_x, self.last_y = None, None

    def clear(self):
        self.canvas.delete("all")
        self.image = Image.new("L", (self.CANVAS_SIZE, self.CANVAS_SIZE), color=0)
        self.draw = ImageDraw.Draw(self.image)
        self.result_label.config(text="여기에 그리고 '예측하기'를 눌러보세요")

    def predict(self):
        img = np.array(self.image)
        if img.max() < 10:
            self.result_label.config(text="먼저 글자를 그려주세요")
            return

        processed = crop_and_pad_image(img)
        input_tensor = processed.astype(np.float32).reshape(1, 28, 28, 1) / 255.0
        prediction = self.model(input_tensor, training=False).numpy()[0]

        pred_letter = LABELS[np.argmax(prediction)]
        top3 = np.argsort(prediction)[::-1][:3]
        top3_txt = ", ".join(f"{LABELS[i]} {prediction[i]*100:.0f}%" for i in top3)
        self.result_label.config(text=f"예측: {pred_letter}   (상위 3개: {top3_txt})")

    def run(self):
        self.root.mainloop()


LetterDrawer(model).run()

**테스트셋에서 임의로 25장 뽑아 확인**

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 10))
for i in range(25):
    idx = np.random.randint(0, len(x_test))
    pred_label = LABELS[np.argmax(model.predict(x_test[idx:idx+1], verbose=0))]
    true_label = LABELS[np.argmax(y_test[idx])]
    plt.subplot(5, 5, i + 1)
    plt.imshow(x_test[idx].reshape(28, 28), cmap='gray')
    color = 'green' if pred_label == true_label else 'red'
    plt.title(f'pred: {pred_label} (true: {true_label})', color=color, fontsize=9)
    plt.axis('off')
plt.suptitle('Random Test Predictions (real EMNIST)', fontsize=16)
plt.tight_layout()
plt.show()